In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import scipy.stats as st
fig = plt.figure()
sns.set_palette("GnBu_d")
sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfTimeMinute.pkl")
df = df[["VIX Natural Log","Negative Natural Log"]]
df.dropna(inplace=True)
df = df[(df.T != 0).any()]
df = df.loc[df["Negative Natural Log"]!=0]
df = df.loc[df["VIX Natural Log"]!=0]
return_df = df
return_df["Time"] = return_df.index
return_df['Time']=return_df['Time'].dt.strftime('%Y-%m-%d %H:%M:%S')
return_df.index = return_df.Time

In [ ]:
events_db = []
for date, row in return_df.T.iteritems():
    print(date)
    new_row = {'event_date': np.datetime64(row["Time"]),'market_ticker':'VIX Natural Log','security_ticker': 'Negative Natural Log'}#row["Company"]}
    events_db.append(new_row)

In [ ]:
event_data = events_db

In [ ]:
stocklist = {"VIX Natural Log","Negative Natural Log"}

In [ ]:
def eventstudy(returndata=return_df, eventdata=event_data, stocklist=stocklist):
    """
    returndata: is a dataframe with the market returns of the different firms
    eventdata: eventdata for the different firms
    stocklist: a list of the firms involved in the analysis
    
    Returns:
    abnreturn: a dictionary of the abnormal returns for each firm in their respective eventwindows -/+20
    """
    abnreturn ={}  # abnormal returns on the event window
    returndata = returndata.reset_index()
    Bse = []
    for stock in stocklist:
        print("Stock="+stock)
        eventindex = int(returndata[returndata['Date'] == str(eventdata.at[stock,'EventDate'])].index.values)
        print(eventindex)
        event_df = returndata.loc[eventindex-20: eventindex+20, ['Date',stock,'RF', 'Mkt_RF','SMB','HML' ]]
        estimation_df  = returndata.loc[eventindex-270: eventindex-21, ["Date",stock, "RF", "Mkt_RF","SMB","HML" ]]
        formula = stock + " - RF ~ Mkt_RF + SMB + HML"
        beta_Mkt = sm.OLS.from_formula(formula, data=estimation_df).fit().params["Mkt_RF"]
        beta_SMB = sm.OLS.from_formula(formula, data=estimation_df).fit().params["SMB"]
        beta_HML = sm.OLS.from_formula(formula, data=estimation_df).fit().params["HML"]
    
        alpha = sm.OLS.from_formula(formula, data=estimation_df).fit().params["Intercept"]
        
        standard_error = sm.OLS.from_formula(formula, data=estimation_df).fit().bse
        
        Bse.append(standard_error)
        print("{}, beta_Mkt= {}, beta_SMB = {}, beta_HML = {}, alpha= {}".format(stock, beta_Mkt, beta_SMB, beta_HML, alpha))
        
        #expected returns for each firm in the estimation window
        expectedreturn_eventwindow = ((event_df[['Mkt_RF']].values * beta_Mkt) + (event_df[['SMB']].values * 
                                       beta_SMB) + (event_df[['HML']].values * beta_HML ) + alpha)
        
        #abnormal returns on the event window - AR
       
        abnormal_return = event_df[stock].values - list(flatten(expectedreturn_eventwindow))
        abnreturn[stock] = abnormal_return
        
        
    abnormalreturns_df = pd.DataFrame(abnreturn)
    abnormalreturns_df.index = abnormalreturns_df.index-20
    return abnormalreturns_df

In [ ]:
eventstudy(return_df, event_data, stocklist)

In [ ]:
abnreturn

In [ ]:
plt.figure(figsize=(24,16))
for i in range(1,17):
    plt.subplot(4,4,i)
    abnormalreturns_df[abnormalreturns_df.columns[i-1]].plot()
    plt.xlabel('Event Window')
    plt.ylabel('Return')
    plt.axhline(y=(np.sqrt((abnormalreturns_df.iloc[:,i-1].std()**2/41))*1.96),color='red',linestyle='--')
    plt.axhline(y=(np.sqrt((abnormalreturns_df.iloc[:,i-1].std()**2/41))*-1.96),color='red',linestyle='--')
    plt.title(abnormalreturns_df.columns[i-1])

In [ ]:
mean_AAR = abnormalreturns_df.mean(axis = 1)
var_AAR = (abnormalreturns_df.std())**2
var_matrix = pd.DataFrame(var_AAR)
var_matrix = var_matrix.T
var_AAR = sum(var_matrix.iloc[0])/16**2
Std_AAR = np.sqrt(var_AAR)
mean_AAR.plot()
plt.axhline(y=Std_AAR*1.96,color='red',linestyle='--')
plt.axhline(y=Std_AAR*-1.96,color='red',linestyle='--')

In [ ]:
def CAR_se(Abnormal_return=abnormalreturns_df, stock_list=stocklist):
    """
    To get the standard error of Cumulative Abnormal Return for each stock
    Input: the Abnormal Return datafram or matrix, a list of company names
    Output: a dataframe of cumulative standard error for each stock
    """
    residual_sigma_single = pd.DataFrame()
    residual_sigma_cum_single = pd.DataFrame()
    resi_single = []
    d = {}
    for x in stocklist:
        resistd = abnormalreturns_df[x].std()/16
        d.update({x:resistd})
        
    residual_sigma_single = pd.DataFrame(d,index=Abnormal_return.index)
    residual_sigma_cum_single = np.sqrt(residual_sigma_single.cumsum())
    se_cum_single = np.sqrt(((residual_sigma_cum_single**2)/16))
    
    return se_cum_single

def CAAR_se(Abnormal_return=abnormalreturns_df, stock_list=stocklist):
    
    """
    To get the standard error of Cumulative Average Abnormal Return
    Input: the Abnormal Return datafram or matrix, a list of company names
    Output: a list of cumulative standard error
    """
    residual_sigma  = pd.DataFrame()
    resi = []
    d = {}
    for x in stocklist:
        resistd = abnormalreturns_df[x].std()/16
        d.update({x:resistd})
        
    residual_sigma = pd.DataFrame(d,index=Abnormal_return.index)
    residual_sigma_cum = np.sqrt(residual_sigma.cumsum())
    se_cum = np.sqrt(((residual_sigma_cum**2)/16).mean(axis=1))
    
    return se_cum

CAR_df = abnormalreturns_df.cumsum()
plt.figure(figsize=(24,16))
for i in range(1,17):
    plt.subplot(4,4,i)
    CAR_df[CAR_df.columns[i-1]].plot()
    plt.plot(se_cum_single.iloc[:,i-1]*1.96, color='red',linestyle='--')
    plt.plot(se_cum_single.iloc[:,i-1]*-1.96, color='red',linestyle='--')
    plt.xlabel('Event Window')
    plt.ylabel('CAR')
    plt.title(CAR_df.columns[i-1])

Var_AAR = ((CAR_df.mean(axis=1))**2)/16
Std_AAR = np.sqrt(Var_AAR)
# CAAR
CAAR = mean_AAR.cumsum()
# Plot CAAR
CAAR.plot(figsize=(12,8))
plt.xlabel("Event Window")
plt.plot(se*1.96, color='red',linestyle='--')
plt.plot(se*-1.96, color='red',linestyle='--')
plt.ylabel("Cumulative Return")
plt.title("Cumulative Average Abnormal Return")